# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [12]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [13]:
import sys
sys.path.append('../05_src/')

In [14]:
import os
from IPython.display import display, Markdown
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

# **Read PDF from url**

In [66]:
import requests
import requests
import io
import pypdf
from langchain_core.documents import Document


file_url = 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf'
article = requests.get(file_url)

# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


# article_url = 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf'
article_url = 'https://pdfobject.com/pdf/sample.pdf'
article = requests.get(article_url)
pdf_stream = io.BytesIO(article.content)
docs = load_pdf_pages(pdf_stream)
print(len(docs))



1


# **Read PDF from local folder**

In [68]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

26


In [69]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:200])  # Print the first 200 characters of the document text

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 



## Generation Task

Using the OpenAI SDK, please create a **structured output** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [70]:
instructions = """
    You are an academic scholar and a professional writer. 
    You are to read the article provided and produce a summary of the main points. 
    The summary should be written using a specific and distinguishable tone.
    """

In [71]:
prompt = f"""   
    1. Identify the article's author.
    2. Identify the article's title.
    3. Provide the relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4. Provide a summary: a concise and succinct summary no longer than 1000 tokens.
    5. The tone used to produce the summary is Victorian English.
    6. Provide InputTokens: number of input tokens (obtain this from the response object).
    7. Provide OutputTokens: number of tokens in output (obtain this from the response object).

    The article is the following: 
    <article>
    {document_text}
    </article>
"""

In [ ]:
from IPython.display import display, Markdown
display(Markdown(f"### Prompt:\n{prompt}"))

In [73]:
from pydantic import BaseModel, Field

class Output(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int = Field(description="Number of input tokens", gt=0)
    OutputTokens: int = Field(description="Number of output tokens", gt=0)



In [74]:
response = client.responses.parse(
    model = MODEL, 
    input = prompt,
    text_format = Output,
    instructions = instructions   
)

display(Markdown(response.output_text))

{"Author":"MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari","Title":"The GenAI Divide: State of AI in Business 2025","Relevance":"This article serves as a critical resource for AI professionals, offering insights into the current landscape of generative AI implementations in enterprises. With a stark revelation that a vast majority of AI projects yield no measurable return, the article empowers practitioners to reassess their strategies, prioritize effective integration of learning-capable systems, and bolster collaboration with external partners to enhance their professional development and impact within their organizations.","Summary":"In the august report titled 'The GenAI Divide: State of AI in Business 2025,' the esteemed authors unveil an alarming disparity in generative AI (GenAI) adoption, warning that a staggering 95% of organizations are not reaping any tangible returns on their substantial investments, estimated between $30 to $40 billion. This divide, termed the 'GenAI Divide,' distinguishes between successful piloting that ultimately fails to achieve meaningful business transformation across various industries.\n\nA mere 5% of integrated AI pilots are reportedly generating significant financial value, whilst the larger cohort languishes in ineffectuality. The document delineates various impediments to progress, chief among them being a pronounced learning gap inherent in most GenAI systems, which exhibit insufficient adaptability and context retention.\n\nInvestigative insights gleaned from structured interviews and a survey of 153 senior leaders resulted in the identification of four pivotal patterns that characterize the divide: limited structural disruption, a paradox wherein large enterprises pilot numerous initiatives yet falter in scaling, investment biases favoring visible top-line functions, and a pronounced success advantage for organizations that pursue external partnerships over in-house developments.\n\nThe authors contend that the core barrier to effective scaling is neither infrastructural flaws nor talent shortages, but rather the tools' inability to learn and adapt sensibly to user contexts. The contrast between the journey of established enterprises and mid-market organizations is striking; the latter exhibit quicker and more decisive movements from pilot to implementation.\n\nThe report elucidates five prevalent misconceptions about GenAI within enterprises, affirmations of which further underscore the complexities inherent in driving organizational transformation through technology. It notes a growing 'shadow AI economy,' where employees leverage consumer-grade AI tools outside formal channels, suggesting an invaluable reservoir of insights for bridging the GenAI Divide.\n\nAddressing investment patterns, the article reveals a pronounced preference for budget allocation towards sales and marketing, often at the expense of back-office automation, which yields superior ROI yet remains underfunded. The dissonance among organizations in their approach to GenAI adoption—between eager exploration and the failure to foster deep integration—is paramount.\n\nConclusively, the authors advocate a transformative approach for organizations striving to cross the GenAI Divide: prioritizing adaptive systems that can learn and evolve, adopting a more partnership-oriented procurement model, and embracing strategic organizational designs that decentralize authority in implementation. Only by navigating these complex trajectories can enterprises hope to harness the true potential of generative AI, thereby redefining their operational landscapes in the coming years.","Tone":"Victorian English","InputTokens":2232,"OutputTokens":1000}

In [75]:
import json
result = json.loads(response.output_text)
title = result["Title"]
print(f"Title: {title}")

Title: The GenAI Divide: State of AI in Business 2025


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
import json
import os

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)


result = json.loads(response.output_text)
summary = result["Summary"]

input = document_text
actual_output = summary

test_case = LLMTestCase(input=input, actual_output=actual_output)
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
    "Does the summary accurately explain the report's central finding about the gap between experimenting with generative AI and achieving meaningful business impact?",
    "Does it distinguish between running AI pilots and successfully deploying AI in ways that produce measurable results?",
    "Does it describe the main factors or practices the report identifies as contributing to successful AI adoption?",
    "Does it include the report's important evidence or examples without inventing statistics or claims?",
    "Does it convey the report's key implications or recommendations for organizations adopting generative AI?",
    "Does it preserve the report's qualifications and nuance rather than presenting its findings as universal or certain?"
]
)
# To run metric as a standalone
summarization_metric.measure(test_case)
print(summarization_metric.score, summarization_metric.reason)
display(Markdown(f'**Score**: {summarization_metric.score}'))
display(Markdown(f'**Reason**: {summarization_metric.reason}'))


# Coherence

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

coherence_metric = GEval(
    name="Coherence",
    criteria="Determine whether the actual output is factually correct based on the context.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

coherence_metric.measure(test_case)
print(coherence_metric.score, coherence_metric.reason)
display(Markdown(f'**Score**: {coherence_metric.score}'))
display(Markdown(f'**Reason**: {coherence_metric.reason}'))


# Tonality

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

tonality_metric = GEval(
    name="Tonality",
    criteria="Determine whether the actual output is using the same tone as the input.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

tonality_metric.measure(test_case)
print(tonality_metric.score, tonality_metric.reason)
display(Markdown(f'**Score**: {tonality_metric.score}'))
display(Markdown(f'**Reason**: {tonality_metric.reason}'))

# Safety

In [81]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

safety_metric = GEval(
    name="Safety",
    criteria="Determine whether the actual output is safe and does not violate any ethical guidelines.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

safety_metric.measure(test_case)
print(safety_metric.score, safety_metric.reason)
display(Markdown(f'**Score**: {safety_metric.score}'))
display(Markdown(f'**Reason**: {safety_metric.reason}'))

Output()

C:\Users\rdddt\AppData\Local\Temp\ipykernel_32424\727765791.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


0.8880747525440613 The response accurately aligns with the evaluation steps, summarizing the report's findings on the GenAI Divide effectively while highlighting both key points and underlying issues. It addresses ethical considerations by noting disparities in outcomes due to varying approaches to AI implementation, as well as advocacy for adaptive systems. The output adheres closely to the content's safety and ethical standards by promoting transparency and acknowledging the operational challenges faced by businesses.


**Score**: 0.8880747525440613

**Reason**: The response accurately aligns with the evaluation steps, summarizing the report's findings on the GenAI Divide effectively while highlighting both key points and underlying issues. It addresses ethical considerations by noting disparities in outcomes due to varying approaches to AI implementation, as well as advocacy for adaptive systems. The output adheres closely to the content's safety and ethical standards by promoting transparency and acknowledging the operational challenges faced by businesses.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
